# 04 — Production Engineering, Evals & Security

Short, offline drills for the production decisions in the authenticated course. Start with the [module index](../course%20content%20HTML/04-production-engineering-evals-security/index.html#lesson-index).

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_support.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from inside the project; study_support.py was not found.")
sys.path.insert(0, str(ROOT))
from study_support import load_anthropic_api_key, messages_create

assert callable(load_anthropic_api_key) and callable(messages_create)
ROOT

## Define done with an eval

Use the cheapest grader that measures the requirement, and calibrate a judge against human labels. [Archive](../course%20content%20HTML/04-production-engineering-evals-security/02-evals-and-judges.html#defining-done-before-you-ship-evals-and-a-calibrated-judge) · [Course S02](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S02)

In [ ]:
import json

def grade_json(text):
    try:
        value = json.loads(text)
        return 10 if set(value) == {"priority", "owner"} else 0
    except json.JSONDecodeError:
        return 0

cases = ['{"priority": "high", "owner": "ops"}', '{"priority": "high"}', '{broken']
assert [grade_json(case) for case in cases] == [10, 0, 0]

human_labels = [9, 5, 2]
judge_scores = [9, 4, 3]
agreement = sum(abs(h - j) <= 1 for h, j in zip(human_labels, judge_scores)) / len(human_labels)
assert agreement == 1.0
agreement

## Localize the regression from a trace

Passing units do not prove the seam. Find the first bad handoff, then add the narrowest test that exercises it. [Archive](../course%20content%20HTML/04-production-engineering-evals-security/03-testing-and-tracing.html#the-pieces-passed-and-the-seam-broke) · [Course S06](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S06)

In [ ]:
trace = [
    {"step": "retrieve", "status": "ok", "output_type": "list[dict]"},
    {"step": "build_prompt", "status": "fail", "expected": "str", "received": "list[dict]"},
    {"step": "model.call", "status": "not_run"},
]

def regression_origin(events):
    return next(event for event in events if event["status"] == "fail")

origin = regression_origin(trace)
targeted_test = "integration" if origin.get("expected") != origin.get("received") else "unit"
assert origin["step"] == "build_prompt" and targeted_test == "integration"
origin, targeted_test

## Retry only what waiting can fix

Malformed output, refusals, and auth failures need repair or escalation; transient service failures get capped backoff. Tool failures must return `is_error`. [Archive](../course%20content%20HTML/04-production-engineering-evals-security/04-failure-handling-and-model-selection.html#surviving-production-failure-tool-errors) · [Course S08](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S08)

In [ ]:
RETRIABLE = {429, 500, 502, 503, 504, 529}

def failure_action(*, status=200, output=None, stop_reason=None, attempt=0, retry_after=None):
    if stop_reason == "refusal":
        return ("fail_fast", "refusal")
    if output is not None:
        try:
            json.loads(output)
        except json.JSONDecodeError:
            return ("repair", "malformed_output")
    if status in RETRIABLE:
        return ("retry", retry_after if retry_after is not None else min(2 ** attempt, 30))
    return ("accept", None) if status == 200 else ("fail_fast", status)

def tool_error(tool_use_id, error):
    return {"type": "tool_result", "tool_use_id": tool_use_id, "is_error": True, "content": str(error)}

assert failure_action(status=429, attempt=3) == ("retry", 8)
assert failure_action(status=529, retry_after=7) == ("retry", 7)
assert failure_action(status=401) == ("fail_fast", 401)
assert failure_action(output="{broken") == ("repair", "malformed_output")
assert failure_action(stop_reason="refusal") == ("fail_fast", "refusal")
assert tool_error("toolu_1", "timeout")["is_error"]

## Hold the budget and choose the model

Instrument each call. Start with the balanced model, move only when the eval justifies it, and fan out only independent work. [Archive](../course%20content%20HTML/04-production-engineering-evals-security/05-cost-and-orchestration.html#keeping-cost-latency-and-reliability-in-budget-across-agents) · [Course S11](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S11) · [Model selection S10A](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S10A)

In [ ]:
def production_plan(*, haiku_meets_bar=False, sonnet_misses_bar=False, independent_parts=False):
    model = "Opus" if sonnet_misses_bar else "Haiku" if haiku_meets_bar else "Sonnet"
    agent_shape = "orchestrator-worker" if independent_parts else "single-agent"
    return {"model": model, "agent_shape": agent_shape}

calls = [
    {"input_tokens": 800, "output_tokens": 120, "latency_ms": 420, "ok": True},
    {"input_tokens": 500, "output_tokens": 90, "latency_ms": 610, "ok": True},
]
observed = {
    "tokens": sum(c["input_tokens"] + c["output_tokens"] for c in calls),
    "latency_ms": sum(c["latency_ms"] for c in calls),
    "error_rate": 1 - sum(c["ok"] for c in calls) / len(calls),
}
assert production_plan(haiku_meets_bar=True)["model"] == "Haiku"
assert production_plan(sonnet_misses_bar=True)["model"] == "Opus"
assert production_plan(independent_parts=False)["agent_shape"] == "single-agent"
assert observed == {"tokens": 1510, "latency_ms": 1030, "error_rate": 0.0}
observed

## Enforce the security boundary

Authentication establishes identity; authorization limits it. Treat fetched content as data, redact PII, reject jailbreaks, and block injected actions outside the prompt. [Archive](../course%20content%20HTML/04-production-engineering-evals-security/06-security.html#securing-the-integration-against-untrusted-input-and-a-regulated-review) · [Course S14](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S14)

In [ ]:
import posixpath
import re
from pathlib import PurePosixPath

ALLOWED_ACTIONS = {"read_input", "write_output"}
OUTPUT_ROOT = PurePosixPath("/workspace/output")

def redact_pii(text):
    return re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "[EMAIL]", text)

def security_gate(identity, action, path, *, source="user", requested_by_content=False, jailbreak=False):
    if not identity:
        return "deny: authentication required"
    if jailbreak:
        return "deny: jailbreak"
    if source == "untrusted_content" and requested_by_content:
        return "deny: prompt injection"
    if action not in ALLOWED_ACTIONS:
        return "deny: authorization"
    if action == "write_output" and not PurePosixPath(posixpath.normpath(path)).is_relative_to(OUTPUT_ROOT):
        return "deny: authorization"
    return "allow"

assert redact_pii("Contact ada@example.com") == "Contact [EMAIL]"
assert security_gate(None, "read_input", "/workspace/input/a").startswith("deny: authentication")
assert security_gate("svc-agent", "delete", "/workspace/output/a") == "deny: authorization"
assert security_gate("svc-agent", "write_output", "/public/exfil.txt") == "deny: authorization"
assert security_gate("svc-agent", "write_output", "/workspace/output/../secrets") == "deny: authorization"
assert security_gate("svc-agent", "write_output", "/workspace/output/a", source="untrusted_content", requested_by_content=True) == "deny: prompt injection"
assert security_gate("svc-agent", "read_input", "/workspace/input/a", jailbreak=True) == "deny: jailbreak"
assert security_gate("svc-agent", "write_output", "/workspace/output/a") == "allow"

## Hands-on exercise

Harden the incidents below. Before running the cell, predict each decision. Then add one new trace-level regression and one new security case. [Cumulative archive](../course%20content%20HTML/04-production-engineering-evals-security/07-cumulative-task.html#cumulative-production-hardening-task-find-the-three-defects-and-explain-each) · [Course S17](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/2gf8jaub9q0dj/Developer_M4_vF2.html#S17)

In [ ]:
exercise = {
    "rate_limit": failure_action(status=429, attempt=2),
    "bad_json": failure_action(output="not-json"),
    "injected_write": security_gate("svc-agent", "write_output", "/public/x", source="untrusted_content", requested_by_content=True),
}
assert exercise == {
    "rate_limit": ("retry", 4),
    "bad_json": ("repair", "malformed_output"),
    "injected_write": "deny: prompt injection",
}
exercise